<a href="https://colab.research.google.com/github/AnshulKumar79/Neer_Nirikshan_Hackathon_Project/blob/main/Neer_Nirikshan_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="FEfRg2eZ9m7JuSwTfuAY")
project = rf.workspace("project-mfzoq").project("potholes-detection-yolov8-rtu6b")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 79.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Potholes-Detection-Yolov8-1 in yolov8:: 100%|██████████| 8006/8006 [00:03<00:00, 2413.35it/s]

Dataset downloaded to: /content/Potholes-Detection-Yolov8-1


In [2]:
# Look at the contents of the data.yaml file
!cat {dataset.location}/data.yaml

names:
- pothole
nc: 1
roboflow:
  license: CC BY 4.0
  project: potholes-detection-yolov8-rtu6b
  url: https://universe.roboflow.com/project-mfzoq/potholes-detection-yolov8-rtu6b/dataset/1
  version: 1
  workspace: project-mfzoq
test: ../test/images
train: ../train/images
val: ../valid/images


In [3]:
import yaml
import os

# Defining the absolute paths to our dataset
dataset_path = dataset.location

data = {
    'train': os.path.join(dataset_path, 'train/images'),
    'val': os.path.join(dataset_path, 'valid/images'),
    'test': os.path.join(dataset_path, 'test/images'),
    'nc': 1,
    'names': ['pothole'] #class_name
}


yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path, 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)

print(f"Updated {yaml_path} successfully!")

Updated /content/Potholes-Detection-Yolov8-1/data.yaml successfully!


In [6]:
!pip install ultralytics
from ultralytics import YOLO

# Loading the pre-trained YOLOv8 Nano model
model = YOLO('yolov8n.pt')

print("Base model loaded. Ready to train on pothole dataset!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Base model loaded. Ready to train on pothole dataset!


In [7]:
results = model.train(
    data=f"{dataset.location}/data.yaml", #The path to our Roboflow data
    epochs=50,                            #No. of training rounds
    imgsz=640,                            #Standard image size for YOLO
    plots=True                            #Generates charts so we can see progress
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Potholes-Detection-Yolov8-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

In [8]:
test_images = os.listdir(f"{dataset.location}/test/images")
sample_path = os.path.join(dataset.location, "test/images", test_images[0])

# Run detection
results = model.predict(source=sample_path, save=True, conf=0.5)

print(f"Results saved! Check 'runs/detect/predict' to see the boxes.")


image 1/1 /content/Potholes-Detection-Yolov8-1/test/images/DASH_CAM_2016_01_29_-42_Miles_of_Potholes-_mp4-284_jpg.rf.7dad53b59660423ad081df0b144cc3e1.jpg: 640x640 1 pothole, 7.8ms
Speed: 3.3ms preprocess, 7.8ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict
Results saved! Check 'runs/detect/predict' to see the boxes.
